In [ ]:
import time
import threading
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
# PL is imported in-case you need to clear the Jupyter cache
# if the BD has been modified. sometimes it wont detect
# AXI IP since it cached an old hwh.

from pynq import Overlay, allocate, PL
from pynq.lib import AxiGPIO
# comment this out if it complains if ur running it the first time.
# PL.reset()
%matplotlib inline

# this is one second due to t = n/Fs. can increase if needed.
BUFFER_SIZE = 48828
# DMA set to transfer 256 beats.
N = 256
REDRAW_EVERY_S = 0.25

overlay = Overlay("active_noise_cancelling.bit")
print(list(overlay.ip_dict.keys()))
dma_err = overlay.err_axi_dma
dma_ref = overlay.ref_axi_dma
# print(list(overlay.ip_dict.keys()))

# this controls MUX select.
gpio_block = AxiGPIO(overlay.ip_dict['sel_gpio'])
gpio_mux = gpio_block.channel1
# mux_state = gpio_mux.read() # comment this out soon.
    # constant NO_NOISE       : std_logic_vector(2 downto 0) := "000"; -- 0
    # constant ANTI_N         : std_logic_vector(2 downto 0) := "001"; -- 1
    # constant WHITE_N        : std_logic_vector(2 downto 0) := "010"; -- 2
    # constant REF_MIC        : std_logic_vector(2 downto 0) := "011"; -- 3
    # constant ERR_MIC        : std_logic_vector(2 downto 0) := "100"; -- 4
# write 1 to the GPIO register, this just sets it to the antinoise output.
gpio_mux.write(1, 0x7)

# ERR and REF MIC DC OFFSET
err_gpio_block = AxiGPIO(overlay.ip_dict['err_gpio'])
err_gpio = err_gpio_block.channel1
# measured offset.
err_gpio.write(1166430-358130, 0xFFFFFF);
ref_gpio_block = AxiGPIO(overlay.ip_dict['ref_gpio'])
ref_gpio = ref_gpio_block.channel1
ref_gpio.write(804550+358130, 0xFFFFFF);

# generate numpy arrays that we shift samples into 256 at a time.
err_plot = np.zeros(BUFFER_SIZE, dtype=np.int32)
ref_plot = np.zeros(BUFFER_SIZE, dtype=np.int32)
plot_lock = threading.Lock()
stop_event = threading.Event()

# function run on seperate thread to receive samples via S2MM to prevent graphing
# from delaying re-arming the DMA
def dma():
    ref_a = allocate(shape=(N,), dtype=np.int32)
    ref_b = allocate(shape=(N,), dtype=np.int32)
    err_a = allocate(shape=(N,), dtype=np.int32)
    err_b = allocate(shape=(N,), dtype=np.int32)


    try:
        dma_ref.recvchannel.transfer(ref_a)
        dma_err.recvchannel.transfer(err_a)
        dma_ref.recvchannel.wait()
        dma_err.recvchannel.wait()

        dma_ref.recvchannel.transfer(ref_b)
        dma_err.recvchannel.transfer(err_b)

        # ping pong between buffers a and b so it can be transfered into one buffer while
        # copying out of the other
        while not stop_event.is_set():
            dma_err.recvchannel.wait()
            dma_ref.recvchannel.wait()

            dma_ref.recvchannel.transfer(ref_a)
            dma_err.recvchannel.transfer(err_a)
            with plot_lock:
                err_plot[:-N] = err_plot[N:]
                err_plot[-N:] = err_b
                ref_plot[:-N] = ref_plot[N:]
                ref_plot[-N:] = ref_b

            ref_a, ref_b = ref_b, ref_a
            err_a, err_b = err_b, err_a
    # when thread closes free allocated numpy buffers.
    finally:
        ref_a.freebuffer()
        ref_b.freebuffer()
        err_a.freebuffer()
        err_b.freebuffer()


worker = threading.Thread(target=dma, daemon=True)
worker.start()

# make numpy array of the required size.
x = np.arange(BUFFER_SIZE)

# setup graphs
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
(err_line,) = ax1.plot(x, err_plot, lw=1)
(ref_line,) = ax2.plot(x, ref_plot, lw=1, color="tab:orange")
ax1.set_title("Error")
ax2.set_title("Reference")
ax2.set_xlabel("Sample")
ax1.set_xlim(0, BUFFER_SIZE)
fig.tight_layout()
plt.close(fig)

err_average = 0;
ref_average = 0;

try:
    # continously update graph every REDRAW_EVERY_S seconds
    while True:
        time.sleep(REDRAW_EVERY_S)

        err_average = np.mean(err_plot)
        ref_average = np.mean(ref_plot)
        
        # copy the plot so other thread can continue
        with plot_lock:
            err_copy = err_plot.copy()
            ref_copy = ref_plot.copy()

        err_line.set_ydata(err_copy)
        ref_line.set_ydata(ref_copy)

        # set y axis scale
        for ax, data in ((ax1, err_copy), (ax2, ref_copy)):
            lo, hi = int(data.min()), int(data.max())
            pad = max(1, (hi - lo) * 0.1)
            ax.set_ylim(-2000000, 2000000)
        
        print(err_average)
        print(ref_average)
            
        clear_output(wait=True)
        display(fig)

except KeyboardInterrupt:
    print("Stopping...")
    stop_event.set()
    worker.join(timeout=1.0)
    from scipy.io import savemat
    with plot_lock:
        savemat("plot_data.mat", {"err_plot": err_plot, "ref_plot": ref_plot})

Stopping...
